# Imports

Here we import the required libraries

In [ ]:
!pip install torchsummary

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
from __future__ import print_function, division

from tqdm.notebook import tqdm
tqdm.pandas()


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from numpy.typing import NDArray
from functools import reduce
from itertools import islice
import wandb
import math
from itertools import chain
import copy
from PIL import Image

import torch
from torch import nn
from torch import Tensor
from torch.optim import Optimizer
import torch.nn.functional as F
import torchvision 
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from torchsummary import summary
# Import albumentations library in order to -use pre-built augmentations
import albumentations as A

from sklearn.model_selection import train_test_split
from multiprocessing import cpu_count

import os
import torch
import os.path as osp
from skimage import io, transform
import matplotlib.pyplot as plt
import typing as ty
import cv2


plt.ion()   # interactive mode

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for root, dirs, filenames in os.walk('/kaggle/input'):
    for i, filepath in enumerate(filenames):
        if i >= 10:
            print()
            break
        print(osp.join(root, filepath))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Creating dataset structure

In [ ]:
torch.manual_seed(32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
test = torch.ones((100, 100)).to(device)
del test
torch.cuda.empty_cache()

This is done in order to control randomness.

In [ ]:
DATA_DIR = '/kaggle/input/competitions/aa-iv-2026-i-object-localization/'
WORK_DIR = '/kaggle/working'
BATCH_SIZE = 32

img_dir = osp.join(DATA_DIR, "cars/cars_resize")
h, w, c = 416, 416, 3 # The heigh, width and number of channels of each image

# Here we map each class to an index from 0 to n_classes - 

df = pd.read_csv(osp.join(DATA_DIR, "train.csv"))

#df.drop(columns=['class_id','class'], inplace=True)
#df.rename(columns = {'id_tipo':'class_id','Tipo':'class'}, inplace=True)

id2obj = df[['class_id','class']].drop_duplicates().reset_index().reset_index().drop(columns=['index','class_id']).rename(columns={'level_0':'class_id'}).set_index('class_id')['class'].to_dict()
obj2id = df[['class_id','class']].drop_duplicates().reset_index().reset_index().drop(columns=['index','class_id']).rename(columns={'level_0':'class_id'}).set_index('class')['class_id'].to_dict()

df["class_id"] = df["class"].map(obj2id)


In [ ]:
df.head()

In [ ]:
list_image = list(df.filename)
data_shape = []
data_dim = []
data_w = []
data_h = []

for i in list_image: ## tqdm(list_image)dura 40 segundos
    ruta_imagen = osp.join(img_dir, i)
    imagen = io.imread(ruta_imagen)
    shapes = imagen.shape
    dimen = imagen.ndim
    imagen = Image.open(ruta_imagen)
    w, h = imagen.size
    data_w.append(w)
    data_h.append(h)
    data_shape.append(shapes)
    data_dim.append(dimen)

data_w_h = pd.DataFrame([list_image,data_shape,data_dim,data_w,data_h]).T.rename(columns={0:'filename',1:'shapes',2:'ndim',3:'w',4:'h'}) 

In [ ]:
data_w_h['shapes'].value_counts()

In [ ]:
data_w_h['ndim'].value_counts()

In [ ]:
df = df.merge(data_w_h[['filename','w','h']], how='inner', on='filename')
num_cols = ['xmin','xmax','ymin','ymax','w','h','class_id']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df.head()

In [ ]:
df['w'].value_counts()

In [ ]:
df['h'].value_counts()

In [ ]:
df.info()

In [ ]:
df['xmin'] = round(df['xmin']/df['w'],2)
df['xmax'] = round(df['xmax']/df['w'],2)
df['ymin'] = round(df['ymin']/df['h'],2)
df['ymax'] = round(df['ymax']/df['h'],2)

columns_f = ['filename', 'xmin','ymin','xmax','ymax','class_id','class','w','h']
df1 = df[columns_f].copy()

train_df, val_df = train_test_split(
    df1, stratify=df1['class_id'], test_size=0.25#, random_state=RANDOM_SEED
)

print(train_df.shape)
print(val_df.shape)

In [ ]:
df

In [ ]:
classes = df["class"].unique()
classes

The training set contians information about the class at each image and the corresponding bounding box.

In [ ]:
df.head()

In [ ]:
train_df['class'].value_counts(1) * 100

In [ ]:
val_df['class_id'].value_counts(1) * 100

But the test set only contains the filename of each image, so we have to generate predictions and send it to the Kaggle competition.

In [ ]:
transform_func_inp_signature = ty.Dict[str, NDArray[np.float64]]
transform_func_signature = ty.Callable[
    [transform_func_inp_signature],
    transform_func_inp_signature
]

class CarsDataset(Dataset):
    """
    Location Cars dataset
    """
    def __init__(
        self, 
        df: pd.DataFrame, 
        root_dir: str, 
        labeled: bool = True,
        transform: ty.Optional[ty.List[transform_func_signature]] = None
    ) -> None:
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.labeled = labeled
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx: int) -> transform_func_signature: 
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        # Read image
        img_name = os.path.join(self.root_dir, self.df.filename.iloc[idx])
        image = io.imread(img_name)
        
        #print(f"Dimensiones originales de la imagen: {image.shape}")  # Agregar para depuración

        if image.ndim == 2:  # Si la imagen está en escala de grises
            image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)  # Convertir a RGB
        elif image.shape[2] == 4:  # Si la imagen es RGBA
            image = image[:, :, :3] 
        
        sample = {'image': image}
        
        if self.labeled:
            # Read labels
            img_class = self.df.class_id.iloc[idx]
            #name_class = self.df['class'].iloc[idx]
            img_bbox = self.df.iloc[idx, 1:5]
            #img_h = self.df.h.iloc[idx]
            #img_w = self.df.w.iloc[idx]

            img_bbox = np.array([img_bbox]).astype('float')
            img_class = np.array([img_class]).astype('int')
            #img_h = np.array([img_h]).astype('int')
            #img_w = np.array([img_w]).astype('int')
            sample.update({'bbox': img_bbox, 'class_id': img_class})#,'name_class':name_class,'w': img_w, 'h': img_h
        
        if self.transform:
            sample = self.transform(sample)
        
        return sample

In [ ]:
def draw_bbox(img, bbox, color):
    xmin, ymin, xmax, ymax = bbox
    img = cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 2)
    return img

def normalize_bbox(bbox, factor : int = 416):
    return list(map(lambda x: int(x * factor), bbox))
'''
def normalize_bbox(bbox, factor_h: int =1 ,factor_w: int=1):
    # Iterar a través de la lista utilizando un bucle for con el índice
    for i in range(len(bbox)):
        if i % 2 == 0:  # Comprobar si el índice es par
            nuevo_valor = int(bbox[i]*factor_w).astype('int')
        else:  # El índice es impar
            nuevo_valor = int(bbox[i]*factor_h).astype('int')
        bbox[i] = int(nuevo_valor).astype('int')
        print(f"Valor: {bbox[i]}, Tipo: {type(bbox[i])}")
    return bbox
    #return list(map(lambda x: int(x * factor), bbox))
'''

def draw_bboxes(imgs, bboxes, colors):
    for i, (img, bbox, color) in enumerate(zip(imgs, bboxes, colors)):
        imgs[i] = draw_bbox(img, bbox, color)
    return imgs

def draw_classes(imgs, classes, colors, origin, offset: int = 5, prefix: str =''):
    for i, (img, class_id, color) in enumerate(zip(imgs, classes, colors)): 
        if type(c)==list:
            name_class_=id2obj[classes[i]]
        else:
            name_class_=id2obj[classes[i][0]]
        imgs[i] = cv2.putText(
            img, f'{prefix}{name_class_}', #class_id.squeeze()
            origin, cv2.FONT_HERSHEY_SIMPLEX, 
            0.4, color, 1, cv2.LINE_AA
        )
    return imgs

def draw_predictions(imgs, classes, bboxes, colors, origin):
    assert all(len(x) > 0 for x in [imgs, classes, bboxes, colors])
    if len(colors) == 1:
        colors = [colors[0] for _ in imgs]
    imgs = draw_bboxes(imgs, bboxes, colors)
    imgs = draw_classes(imgs, classes, colors, origin)
    return imgs

In [ ]:
train_root_dir = osp.join(DATA_DIR, "cars/cars_resize")#, "train"
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

num_imgs = 6
start_idx = 0

samples = [train_ds[i] for i in range(start_idx, num_imgs)]

imgs = [s['image'] for s in samples]
bboxes = [normalize_bbox(s['bbox'].squeeze()) for s in samples]
classes = [s['class_id'] for s in samples]

imgs = draw_predictions(imgs, classes, bboxes, [(0, 150, 0)], (5, 10))#(150, 10)

fig = plt.figure(figsize=(30, num_imgs))

for i, img in enumerate(imgs):
    fig.add_subplot(1, num_imgs, i+1)
    plt.imshow(img)

plt.show()

# Transfer learning

In our case, we are interested in getting a pretrained-model to use it as a backbone that has been trained in other tasks. For example, if we want to use vgg16 as our backbone, we would not need the last classification module and keep the averaged pool module as the source of features to perform both tasks

In [ ]:
# =========================
# VGG16 Feature Extractor
# - Freeze first 4 blocks (features[:24]), unfreeze last block (features[24:])
# - Reduced dropout to 0.3 to avoid underfitting
# =========================

import torch
from torch import nn
from torchvision.models import vgg16, VGG16_Weights

# ---- dispositivo (asegúrate de tener 'device' definido)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FeatureExtractor(nn.Module):
    def __init__(self, backbone: nn.Module, freeze_backbone: bool = True, dropout_p: float = 0.3):
        super().__init__()

        # VGG16 conv blocks
        self.features = backbone.features  # nn.Sequential
        # Average pooling layer
        self.pooling = backbone.avgpool
        # Flatten + Dropout
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=dropout_p)

        if freeze_backbone:
            # Freeze only the first 4 blocks (features[:24]), keep block 5 (features[24:]) trainable
            for p in self.features[:24].parameters():
                p.requires_grad = False
            for p in self.features[24:].parameters():
                p.requires_grad = True
            for p in self.pooling.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pooling(x)
        x = self.flatten(x)
        x = self.dropout(x)
        return x

# ---- cargar pesos modernos (elimina warnings)
weights = VGG16_Weights.DEFAULT
vgg16_model = vgg16(weights=weights)

# ---- crear extractor: block 5 descongelado, dropout reducido
pretrained_model = FeatureExtractor(vgg16_model, freeze_backbone=True, dropout_p=0.3).to(device)

print(pretrained_model)


In [ ]:
summary(pretrained_model, (3, 416, 416))

In [ ]:
vgg16_model

In [ ]:
samples[2]['image']

# Image normalization

In [ ]:
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

means = np.zeros(3)
stds = np.zeros(3)
n_images = 0

for x in train_ds:
    img = x['image']#astype(np.float32)  # Asegúrate de que la imagen está en float para cálculos precisos
    n_images += 1

    for channel in range(3):
        channel_pixels = img[..., channel]
        # Acumular la suma y suma de cuadrados para calcular la media y desviación estándar
        means[channel] += np.mean(channel_pixels)
        stds[channel] += np.std(channel_pixels)

# Calcular la media y desviación estándar final
means /= n_images
stds /= n_images

In [ ]:
print(means)
print(stds)

In [ ]:
!wget https://upload.wikimedia.org/wikipedia/commons/9/99/Gioconda_%28copia_del_Museo_del_Prado_restaurada%29.jpg

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

img_filename = "Gioconda_(copia_del_Museo_del_Prado_restaurada).jpg"

print("Directorio actual:", os.getcwd())
print("¿Existe el archivo aquí?", os.path.exists(img_filename))

monalisa_bgr = cv2.imread(img_filename)

if monalisa_bgr is None:
    raise FileNotFoundError(
        f"No pude leer la imagen con cv2.imread('{img_filename}'). "
        "Revisa que el nombre y la ruta sean correctos"
        "y que el archivo exista en el directorio actual."
    )

monalisa_rgb = cv2.cvtColor(monalisa_bgr, cv2.COLOR_BGR2RGB)
print("Shape:", monalisa_rgb.shape)

plt.imshow(monalisa_rgb)
plt.axis("off")
plt.show()

In [ ]:
img_filename = osp.join(DATA_DIR, "cars/cars_resize",'6_train.jpg')
img1 = cv2.imread(img_filename)
#img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

print(img1.shape)
print(img1.transpose((2,0,1)).shape)
img1[..., 0] = 0
img1[..., 1] = 0

plt.imshow(img1)

# Image transforms

A recommended library to do image augmentation is https://albumentations.ai/docs/3-basic-usage/bounding-boxes-augmentations/


In [ ]:
class ToTensor(object):
    """Convert ndarrays in sample to Tensors."""

    def __call__(self, sample):
        image = sample['image']

        # swap color axis because
        # numpy image: H x W x C (0,1,2)
        # torch image: C x H x W
        image = image.transpose((2, 0, 1))
        image = torch.from_numpy(image).float()
        sample.update({'image': image})
        return sample

class Normalizer(object):
    
    def __init__(self, stds, means):
        """
        Arguments:
        
            stds: array of length 3 containing the standard deviation of each channel in RGB order.
            means: array of length 3 containing the means of each channel in RGB order.
        """
        self.stds = stds
        self.means = means
    
    def __call__(self, sample):
        """
        Sample: a dicitonary containing:
            image: sample image in format (C, H, W)
        Returns:
            the image in (C, H, W) format with the channels normalized.
        """
        image = sample['image']
        
        for channel in range(3):
            image[channel] = (image[channel] - means[channel]) / stds[channel]

        sample['image'] = image
        return sample

class TVTransformWrapper(object):
    """Torch Vision Transform Wrapper
    """
    def __init__(self, transform: torch.nn.Module):
        self.transform = transform
        
    def __call__(self, sample):
        sample['image'] = self.transform(sample['image'])
        return sample

class AlbumentationsWrapper(object):
    
    def __init__(self, transform):
        self.transform = transform
    
    def __call__(self, sample):
        transformed = self.transform(
            image=sample['image'], 
            bboxes=sample['bbox'],
            #category_ids=sample['class_id']
        )
        sample['image'] = transformed['image']
        sample['bbox'] = np.array(transformed['bboxes'])
        return sample

In [ ]:
common_transforms = [    
    ToTensor(),
    Normalizer(
        means=means,
        stds=stds,
    )
]

train_data_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=(-20,20), p=0.5),
    A.Resize(height=416, width=416, p=1) # p=1 means always apply
    ],
    bbox_params=A.BboxParams(
        format='albumentations', 
        label_fields=[],#'category_ids'
    )
)

train_transforms = torchvision.transforms.Compose(
    [
        AlbumentationsWrapper(train_data_augmentations),
    ] + common_transforms
)

eval_transforms = torchvision.transforms.Compose(common_transforms)


In [ ]:
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

x = next(iter(train_ds))
x_transformed = copy.deepcopy(x)
x_transformed = train_transforms(x_transformed)

original_img = x['image']
transformed_img = x_transformed['image'].numpy().transpose(1, 2, 0)

original_img = draw_bbox(
    original_img,
    normalize_bbox(x['bbox'].squeeze()),
    (0, 255, 0)
)

transformed_img = draw_bbox(
    transformed_img,
    normalize_bbox(x_transformed['bbox'].squeeze()),
    (0, 255, 0)
)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

axes[0].imshow(original_img)
axes[0].set_title('Original digit')

axes[1].imshow(transformed_img)
axes[1].set_title('Transformed digit')

plt.show()

In [ ]:
train_ds = CarsDataset(df, root_dir=train_root_dir, transform=train_transforms)
train_data = torch.utils.data.DataLoader(train_ds, batch_size=16)

for x in train_data:
    print(x['image'].size())
    break


In [ ]:
def get_output_shape(model: nn.Sequential, image_dim: ty.Tuple[int, int, int]):
    return model(torch.rand(*(image_dim)).to(device)).data.shape

class Model(nn.Module):
    def __init__(self, input_shape: ty.Tuple[int, int, int] = (3, 416, 416), n_classes: int = 22):
        """
        Model with one input (image) and two outputs: 
            1. Digit classification (classification).
            2. Bounding box prediction (regression). 
        
        Arguments:
            input_shape: input shape of the image in format (C, H, W)
            n_classes: number of classes to perfrom classification with
            
        Attributes:
            backbone: ConvNet that process the image and 
            returns a flattened vector with the information of the 
            activations.
            
            cls_head: MLP that receives the flattened input from the backbone 
            and predicts the classification logits for the classes (classficiation task).
            
            reg_head: MLP that receives the flattened input from the backbone 
            and predicts the coordinates of the predicted bounding box (regression task). 
        """
        super().__init__()
        
        self.input_shape = input_shape
        
        # When doing transfer learning, use pretrained model instead of custom backbone
        self.backbone = pretrained_model
        
        backbone_output_shape = get_output_shape(self.backbone, [1, *input_shape])
        backbone_output_features = reduce(lambda x, y: x*y, backbone_output_shape)
        
        self.cls_head = nn.Sequential(
            nn.Linear(in_features=backbone_output_features, out_features=768),
            nn.ReLU(),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes)
        )
        
        self.reg_head = nn.Sequential(
            nn.Linear(in_features=backbone_output_features, out_features=768),
            nn.ReLU(),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        )

    def forward(self, x: Tensor) -> ty.Dict[str, Tensor]:
        features = self.backbone(x)
        cls_logits = self.cls_head(features)
        pred_bbox = self.reg_head(features)
        predictions = {'bbox': pred_bbox, 'class_id': cls_logits}
        return predictions

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print('image', x['image'].size())
model = Model(input_shape=(3, 416, 416), n_classes=22).to(device)
x['image'] = x['image'].to(device)
preds = model(x['image'])
preds

# Metrics

In [ ]:
def iou(y_true: Tensor, y_pred: Tensor):
    pairwise_iou = torchvision.ops.box_iou(y_true.squeeze(), y_pred.squeeze())
    result = torch.trace(pairwise_iou) / pairwise_iou.size()[0]
    return result

In [ ]:
def accuracy(y_true: Tensor, y_pred: Tensor):
    pred = torch.argmax(y_pred, axis=-1)
    y_true = y_true.squeeze()
    correct = torch.eq(pred, y_true).float()
    total = torch.ones_like(correct)
    result = torch.divide(torch.sum(correct), torch.sum(total))
    return result

# Loss fn

In [ ]:
def loss_fn(y_true, y_preds, alpha: float = 0.5):
    cls_y_true, cls_y_pred = y_true['class_id'].long(), y_preds['class_id'].float().unsqueeze(-1)
    reg_y_true, reg_y_pred = y_true['bbox'].float().squeeze(), y_preds['bbox'].float().squeeze()
    
    cls_loss = F.cross_entropy(cls_y_pred, cls_y_true)
    
    # Smooth L1 is more robust to outliers than MSE for bbox regression
    reg_loss = F.smooth_l1_loss(reg_y_pred, reg_y_true)
    # Adds weights to both tasks
    total_loss = (1 - alpha) * cls_loss + alpha * reg_loss
    return dict(loss=total_loss, reg_loss=reg_loss, cls_loss=cls_loss)


# Callbacks

In [ ]:
def printer(logs: ty.Dict[str, ty.Any]):
    # print every 10 steps
    if logs['iters'] % 10 != 0:
        return
    print('Iteration #: ',logs['iters'])
    for name, value in logs.items():
        if name == 'iters':
            continue
        
        if type(value) in [float, int]:
            value = round(value, 4)
        elif type(value) is torch.Tensor:
            value = torch.round(value, decimals=4)
        
        print(f'\t{name} = {value}')
    print()

# Training loop

In [ ]:
def evaluate(
    logs: ty.Dict[str, ty.Any], 
    labels: ty.Dict[str, Tensor],
    preds: ty.Dict[str, Tensor],
    eval_set: str,
    metrics: ty.Dict[str, ty.Callable[[Tensor, Tensor], Tensor]],
    losses: ty.Optional[ty.Dict[str, Tensor]] = None,
) -> ty.Dict[str, ty.Any]:
    
    if losses is not None:
        for loss_name, loss_value in losses.items():
            logs[f'{eval_set}_{loss_name}'] = loss_value
    
    for task_name, label in labels.items():
        for metric_name, metric in metrics[task_name]:
            value = metric(label, preds[task_name])
            logs[f'{eval_set}_{metric_name}'] = value
            
    return logs

def step(
    model: Model, 
    optimizer: Optimizer, 
    batch: CarsDataset,
    loss_fn: ty.Callable[[ty.Dict[str, torch.Tensor]], torch.Tensor],
    device: str,
    train: bool = False,
) -> ty.Tuple[ty.Dict[str, Tensor], ty.Dict[str, Tensor]]:
    
    if train:
        optimizer.zero_grad()
    
    img = batch.pop('image').to(device)
    
    for k in list(batch.keys()):
        batch[k] = batch[k].to(device)
    
    preds = model(img.float())
    losses = loss_fn(batch, preds)
    final_loss = losses['loss']
    
    if train:
        final_loss.backward()
        optimizer.step()
    
    return losses, preds


def train(
    model: Model, 
    optimizer: Optimizer, 
    dataset: DataLoader,
    eval_datasets: ty.List[ty.Tuple[str, DataLoader]],
    loss_fn: ty.Callable[[ty.Dict[str, torch.Tensor]], torch.Tensor],
    metrics: ty.Dict[str, ty.Callable[[Tensor, Tensor], Tensor]],
    callbacks: ty.List[ty.Callable[[ty.Dict[ty.Any, ty.Any]], None]],
    device: str,
    train_steps: int = 1000,
    eval_steps: int = 10,
    scheduler=None,
) -> Model:
    # Send model to device (GPU or CPU)
    model = model.to(device)
    iters = 0
    iterator = iter(dataset)
    assert train_steps > eval_steps, 'Train steps should be greater than the eval steps'
    
    while iters <= train_steps:
        model.train()
        logs = dict()
        logs['iters'] = iters
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(dataset)
            batch = next(iterator)
        # Send batch to device 
        losses, preds = step(model, optimizer, batch, loss_fn, device, train=True)
        logs = evaluate(logs, batch, preds, 'train', metrics, losses)
        
        # Eval every eval_steps iterations
        if iters % eval_steps == 0:        
            model.eval()
            with torch.no_grad():
                for name, eval_dataset in eval_datasets:
                    for batch in eval_dataset:
                        losses, preds = step(model, optimizer, batch, loss_fn, device, train=False)            
                        logs = evaluate(logs, batch, preds, name, metrics, losses)

            # Step scheduler based on val_loss
            if scheduler is not None:
                val_loss = logs.get('val_loss', None)
                if val_loss is not None:
                    scheduler.step(val_loss)
        
        for callback in callbacks:
            callback(logs)
        
        iters += 1
    
    return model


# Run

In [ ]:
# Hparams
batch_size = 16
lr = 1e-4  # Lower lr: backbone block 5 is now unfrozen

# Data
train_ds = CarsDataset(train_df, root_dir=train_root_dir, transform=train_transforms)
val_ds = CarsDataset(val_df, root_dir=train_root_dir, transform=eval_transforms)

train_data = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=cpu_count())
val_data = DataLoader(val_ds, batch_size=batch_size, num_workers=cpu_count())

# Model
model = Model().to(device)
summary(model, model.input_shape)


In [ ]:
# Optimizer + Scheduler
optimizer = torch.optim.Adam(lr=lr, params=model.parameters())
# Reduce lr by 0.5 if val_loss doesn't improve for 50 eval steps
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=50, verbose=True
)

model = train(
    model,
    optimizer,
    train_data,
    eval_datasets=[('val', val_data)],
    loss_fn=loss_fn,
    metrics={
        'bbox': [('iou', iou)],
        'class_id': [('accuracy', accuracy)]
    },
    callbacks=[printer],
    device=device,
    train_steps=3000,
    eval_steps=10,
    scheduler=scheduler,
)


In [ ]:
model.parameters()

In [ ]:
num_imgs = 5
ncols = 5
nrows = math.ceil(num_imgs / ncols)

start_idx = 0

inference_ds = CarsDataset(val_df.iloc[start_idx:start_idx+num_imgs], root_dir=train_root_dir)
inference_data = DataLoader(inference_ds, batch_size=num_imgs, num_workers=1, shuffle=False)
inference_batch = next(iter(inference_data))
inference_imgs = np.empty((num_imgs, 3, 416, 416))

transform = eval_transforms

for i, img in enumerate(inference_batch['image']):
    inference_imgs[i] = transform(dict(image=img.numpy()))['image'].numpy()

preds = model(torch.tensor(inference_imgs).float().to(device))

samples = [inference_ds[i] for i in range(start_idx, num_imgs)]

imgs = [s['image'] for s in samples]
bboxes = [normalize_bbox(s['bbox'].squeeze()) for s in samples]
classes = [s['class_id'] for s in samples]

pred_bboxes = preds['bbox'].detach().cpu().numpy()
pred_bboxes = [normalize_bbox(bbox) for bbox in pred_bboxes]
pred_classes = preds['class_id'].argmax(-1).detach().cpu().numpy()

In [ ]:
# Green: ground truth
imgs = draw_predictions(imgs, classes, bboxes, [(0, 150, 0)], (5, 10))
# Red: predicted
pred_classes_ = []
for i in range(num_imgs):
    temp = np.array([pred_classes[i]])
    pred_classes_.append(temp)
imgs = draw_predictions(imgs, pred_classes_, pred_bboxes, [(200, 0, 0)], (150, 10))

fig, axes = plt.subplots(nrows=nrows, ncols=num_imgs // nrows, figsize=(30, 30))

for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!wget https://autoportal.com/cdn/media/cache/image_300x225/img/new-cars-gallery/chevrolet/camaro/photo34/chevrolet-camaro-79ed1401.jpg
!ls

In [ ]:
pickle_rick_img = io.imread('chevrolet-camaro-79ed1401.jpg')
pickle_rick_img = cv2.resize(pickle_rick_img, (h, w))
pickle_rick_img_=np.expand_dims(pickle_rick_img.transpose(2,0,1), 0)

In [ ]:
plt.imshow(pickle_rick_img)

In [ ]:
pickle_rick_img.shape, pickle_rick_img_.shape

In [ ]:
preds = model(torch.tensor(pickle_rick_img_).float().to(device))
pred_classes = preds['class_id'].argmax(-1).detach().cpu().numpy()
pred_bboxes = preds['bbox'].detach().cpu().numpy().tolist()
pred_class = id2obj[pred_classes[0]]
xmin=int(pred_bboxes[0][0]*w)
ymin=int(pred_bboxes[0][1]*h)
xmax=int(pred_bboxes[0][2]*w)
ymax=int(pred_bboxes[0][3]*h)
pickle_rick_img_ = cv2.rectangle(pickle_rick_img, (xmin, ymin), (xmax, ymax), (255, 0, 0))
pickle_rick_img_ = cv2.putText(pickle_rick_img_, pred_class, (xmin, ymin-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
plt.figure(figsize=(5, 5))
plt.imshow(pickle_rick_img_)
#plt.axis("off")
plt.show()

In [ ]:
# Save the model to disk
torch.save(model, 'pretrained_model.pth')

# Submission

In [ ]:
# Perform inference on cpu in order to avoid memory problems 
device = 'cuda'
model = model.to(device)

test_root_dir = osp.join(DATA_DIR, "cars/cars_resize")
test_df = pd.read_csv(osp.join(DATA_DIR, "test.csv"))

test_ds = CarsDataset(test_df, root_dir=test_root_dir, labeled=False, transform=eval_transforms)
test_data = DataLoader(test_ds, batch_size=1, num_workers=cpu_count(), shuffle=False)

class_preds = []
bbox_preds = []

for batch in test_data:
    batch_preds = model(batch['image'].float().to(device))
    
    class_pred = batch_preds['class_id'].argmax(-1).detach().cpu().numpy()
    bbox_pred = batch_preds['bbox'].detach().cpu().numpy()
    
    class_preds.append(class_pred.squeeze())
    bbox_preds.append(bbox_pred.squeeze())

In [ ]:
class_preds = np.array(class_preds)
bbox_preds = np.array(bbox_preds)

In [ ]:
submission = pd.DataFrame(
    index=test_df.filename,
    data={
        'class': class_preds
    }
)

submission["xmin"] = bbox_preds[:, 0]*w
submission["ymin"] = bbox_preds[:, 1]*h
submission["xmax"] = bbox_preds[:, 2]*w
submission["ymax"] = bbox_preds[:, 3]*h

submission

In [ ]:
submission['class']=submission['class'].replace(id2obj)

In [ ]:
submission['class'].value_counts()

In [ ]:
submission

In [ ]:
submission.to_csv('submission.csv')